In [0]:
# Bronze — Source 02: Debezium CDC
# CDC envelope: {before, after, op, source, ts_ms}
# op: c=create, u=update, d=delete, r=read(snapshot)
import sys
sys.path.append('/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared')
from bronze_utils import get_watermark, update_watermark

RAW_BUCKET = 's3://ecommerce-lakehouse-467091806172-raw-01'
SOURCE = '02_debezium_cdc'
TARGET_TABLE = 'bronze.src_02_cdc.events'
MERGE_KEY = 'cdc_event_id'
PATH = f'{RAW_BUCKET}/source=02_debezium_cdc/debezium.ecommerce.*/year=*/month=*/day=*/'


In [0]:
from pyspark.sql.functions import col, lit, max as spark_max, from_json, concat_ws, md5
from pyspark.sql.types import *
import json as json_lib

watermark = get_watermark(spark, SOURCE)
print(f'[{SOURCE}] Watermark: {watermark}')

raw_df = spark.read.text(PATH) \
    .filter(col('_metadata.file_modification_time') > lit(watermark))

if raw_df.count() == 0:
    print(f'[{SOURCE}] No new files — skipping')
    dbutils.notebook.exit('No new data')

from pyspark.sql.functions import udf

@udf(returnType=StringType())
def unwrap_json(s):
    if s is None: return None
    try:
        inner = json_lib.loads(s)
        if isinstance(inner, str): return inner
        import json as j; return j.dumps(inner)
    except: return s

# CDC envelope schema
after_schema = StructType([
    StructField('order_id', LongType()),
    StructField('customer_id', LongType()),
    StructField('status', StringType()),
    StructField('total_pence', LongType()),
    StructField('placed_at', StringType()),
    StructField('updated_at', StringType()),
])

schema = StructType([
    StructField('op', StringType()),
    StructField('ts_ms', LongType()),
    StructField('after', after_schema),
    StructField('before', after_schema),
    StructField('source', StructType([
        StructField('table', StringType()),
        StructField('lsn', LongType()),
        StructField('txId', LongType()),
    ])),
])

df = raw_df \
    .withColumn('unwrapped', unwrap_json(col('value'))) \
    .withColumn('parsed', from_json(col('unwrapped'), schema)) \
    .select(
        col('parsed.op').alias('cdc_op'),
        col('parsed.ts_ms').alias('cdc_ts_ms'),
        col('parsed.source.table').alias('cdc_table'),
        col('parsed.source.lsn').alias('cdc_lsn'),
        col('parsed.after').alias('after'),
        col('parsed.before').alias('before'),
    ) \
    .withColumn('cdc_event_id', md5(concat_ws('|', col('cdc_ts_ms').cast('string'), col('cdc_lsn').cast('string'), col('cdc_op')))) \
    .filter(col('cdc_op').isNotNull())

row_count = df.count()
print(f'[{SOURCE}] {row_count} CDC events parsed')

ops = df.groupBy('cdc_op').count().collect()
for r in ops:
    print(f'  op={r["cdc_op"]}: {r["count"]} events')

spark.sql('CREATE SCHEMA IF NOT EXISTS bronze.src_02_cdc')

if spark.catalog.tableExists(TARGET_TABLE):
    from delta.tables import DeltaTable
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(df.alias('s'), f't.{MERGE_KEY} = s.{MERGE_KEY}') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    df.write.format('delta').mode('overwrite') \
        .option('mergeSchema', 'true').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

latest_ts = raw_df.select(spark_max('_metadata.file_modification_time')).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, row_count)
print(f'Watermark updated to {latest_ts}')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'{TARGET_TABLE}: {count} rows')
spark.sql(f"SELECT * FROM bronze.pipeline.watermarks WHERE source = '{SOURCE}'").show()
print('\nSample CDC events:')
spark.sql(f'SELECT cdc_op, cdc_table, cdc_ts_ms FROM {TARGET_TABLE} LIMIT 5').show()
